# Exercise 55 — Selection: `.loc`, `.iloc`, boolean masks

## Concept

pandas has three (sometimes confusing) ways to select rows and columns:

### `df["col"]` / `df[["a","b"]]` — column selection

```python
df["name"]              # one column (Series)
df[["name", "age"]]     # several columns (DataFrame)
```

### `.loc[rows, cols]` — **label-based**

Uses index labels and column names. Slices are **inclusive** at both ends.

```python
df.loc[0]                 # row with label 0
df.loc[0, "name"]         # single cell
df.loc[0:3, "name"]       # rows 0..3 INCLUSIVE, name column
df.loc[:, ["name", "age"]]
df.loc[mask, "amount"]    # boolean mask + column
```

### `.iloc[rows, cols]` — **integer position-based**

Slices behave like Python — stop is exclusive.

```python
df.iloc[0]                # first row
df.iloc[0:3]              # rows at positions 0, 1, 2 (NOT 3)
df.iloc[-1]               # last row
df.iloc[:, 0]             # first column (Series)
```

### Boolean masks

Use element-wise comparisons to filter rows. Combine with `&` (and), `|` (or), `~` (not). **Always parenthesize** each comparison — operator precedence will surprise you otherwise.

```python
mask = (df["amount"] > 100) & (df["paid"] == True)
df[mask]                       # rows where mask is True
df.loc[mask, ["id", "amount"]]
```

### Chained indexing — avoid

`df[df.amount > 100]["region"] = "N"` looks innocent but is a famous trap (`SettingWithCopyWarning`, may or may not assign). Use a single `.loc[mask, col] = value` instead.

## Setup

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "order_id":   [1, 2, 3, 4, 5, 6, 7, 8],
    "customer":   ["Alice", "Bob", "Alice", "Carol", "Bob", "Alice", "Carol", "Diana"],
    "region":     ["North", "South", "North", "South", "North", "South", "North", "East"],
    "amount":     [120.50, 45.00, 300.75, 89.99, 210.00, 55.25, 430.10, 12.00],
    "paid":       [True, True, False, True, True, False, True, True],
})
print(df)


## Task 1 — First and last rows by position

Write two functions returning the first and last row as a Series.

```python
def first_row(df: pd.DataFrame) -> pd.Series: ...
def last_row(df: pd.DataFrame) -> pd.Series: ...
```

Use `.iloc` (position-based) — that way the functions work even if the index is exotic (e.g., not 0..n-1).

In [ ]:
import pandas as pd

def first_row(df: pd.DataFrame) -> pd.Series:
    pass  # TODO

def last_row(df: pd.DataFrame) -> pd.Series:
    pass  # TODO

fr = first_row(df)
assert isinstance(fr, pd.Series)
assert fr["order_id"] == 1
assert fr["customer"] == "Alice"

lr = last_row(df)
assert lr["order_id"] == 8
assert lr["customer"] == "Diana"
print("ok")


## Task 2 — Filter with a boolean mask

Return rows where `amount` is greater than `threshold`.

```python
def expensive(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    ...
```

Expected: `expensive(df, 200)` has 2 rows (`order_id` 3 and 7). Note: 210.00 is included with `>` only if you use `>=`, here we use **strictly greater than** — so 210.00 IS included with `>`.

In [ ]:
import pandas as pd

def expensive(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    pass  # TODO

result = expensive(df, 200)
assert len(result) == 3
assert set(result["order_id"]) == {3, 5, 7}    # 300.75, 210.00, 430.10
print("ok")


## Task 3 — Compound mask

Return rows that satisfy **both** conditions: `amount > 100` AND `paid is True`. Then drop unused columns — return only `["order_id", "customer", "amount"]`.

```python
def big_paid_orders(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

**Always parenthesize each comparison** when combining with `&`/`|`. `df.amount > 100 & df.paid` is a parser bug waiting to happen.

In [ ]:
import pandas as pd

def big_paid_orders(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = big_paid_orders(df)
assert list(result.columns) == ["order_id", "customer", "amount"]
assert set(result["order_id"]) == {1, 5, 7}   # >100 and paid=True
print("ok")


## Task 4 — Filter using `.isin`

Return rows whose `region` is in a given set of allowed regions.

```python
def rows_in_regions(df: pd.DataFrame, regions: list[str]) -> pd.DataFrame:
    ...
```

Hint: `df["region"].isin(regions)` produces the boolean mask. This is the idiomatic way to filter on "value in collection" — readable and fast on large frames.

Expected: `rows_in_regions(df, ["North", "East"])` returns 5 rows.

In [ ]:
import pandas as pd

def rows_in_regions(df: pd.DataFrame, regions: list[str]) -> pd.DataFrame:
    pass  # TODO

result = rows_in_regions(df, ["North", "East"])
assert len(result) == 5
assert set(result["region"]) <= {"North", "East"}
print("ok")


## Task 5 — Safe assignment with `.loc`

Given the DataFrame, set `paid = True` for all rows where `amount < 50`. **Do not** mutate the input — make a copy first.

```python
def force_paid_small(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Use the **single-statement** `.loc[mask, col] = value` pattern. **Do not** use chained indexing (`df[mask]["paid"] = True`) — that's the trap.

Expected:
- `force_paid_small(df).loc[df["amount"] < 50, "paid"]` is all `True`
- The input `df` is unchanged

In [ ]:
import pandas as pd

def force_paid_small(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

original_paid = df["paid"].copy()
result = force_paid_small(df)

# rows with amount < 50 are now all True
small = result[result["amount"] < 50]
assert small["paid"].all()

# input not mutated
assert df["paid"].tolist() == original_paid.tolist()
print("ok")
